# Lab 01 — Konsep Dasar Apache Spark
**Arsitektur, partition, lazy evaluation, DAG, job/stage/task, dan Spark UI**

Notebook ini dirancang untuk environment pelatihan: Spark 3.5.7 standalone, Jupyter, 2 worker, masing-masing 2 core / 2 GB.  
**Prinsip lab:** prediksi → jalankan → amati → jelaskan. Jangan hanya mengejar output akhir.

## Capaian praktik
- Mengidentifikasi Driver, Master/Cluster Manager, Worker, Executor, Partition, Task, Stage, dan Job.
- Membuktikan lazy evaluation dan membedakan transformation vs action.
- Menghubungkan `explain()` dengan Jobs/Stages/Tasks di Spark UI.
- Mengamati pengaruh jumlah partition terhadap paralelisme.

In [58]:
spark.stop()

In [59]:
from pyspark.sql import SparkSession, functions as F, types as T
import time, json, os, shutil

spark = (
    SparkSession.builder
    .appName("spark-training-lab-01")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("App name      :", spark.sparkContext.appName)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Spark version : 3.5.7
Master        : spark://spark-master:7077
App name      : spark-training-lab-01
Default parallelism: 2


## Lab 1 — Kenali cluster yang sedang dipakai (15–20 menit)
**Sebelum menjalankan kode, prediksi:** berapa core total cluster dan di mana Driver berjalan?

Buka juga:
- Master UI: http://localhost:8080
- Worker UI: http://localhost:8081 dan http://localhost:8082
- Application UI: http://localhost:4040 (selama SparkSession aktif)

Catat jumlah worker, core, memory, dan executor yang terlihat.

In [2]:
print("Application ID:", spark.sparkContext.applicationId)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("UI URL dari SparkContext:", spark.sparkContext.uiWebUrl)

Application ID: app-20261110024957-0010
Default parallelism: 4
UI URL dari SparkContext: http://c6489f47d953:4040


**Observasi peserta**
1. Apakah jumlah core cluster sama dengan jumlah core yang digunakan aplikasi?
2. Apakah Driver merupakan Worker?
3. Mengapa aplikasi kedua dapat menunggu resource walaupun cluster masih hidup?

## Lab 2 — Partition → Task (20 menit)
Buat DataFrame dengan 4 partition, lalu ubah menjadi 8. Amati `count()` pada Spark UI.

In [48]:
df2 = spark.range(0, 1_000_000, 1, 2)

print("df2 partitions:", df2.rdd.getNumPartitions())

df2 partitions: 2


In [49]:
print("Jumlah baris:", df2.count())

Jumlah baris: 1000000


In [52]:
df4 = spark.range(0, 1_000_000, 1, 4)

print("df4 partitions:", df4.rdd.getNumPartitions())

df4 partitions: 4


In [53]:
print("Jumlah baris:", df4.count())

Jumlah baris: 1000000


In [56]:
df8 = spark.range(0, 1_000_000, 1, 8)

print("df8 partitions:", df8.rdd.getNumPartitions())

df8 partitions: 8


In [57]:
print("Jumlah baris:", df8.count())

[Stage 0:>                                                          (0 + 4) / 8]

Jumlah baris: 1000000


In [8]:
# Lihat jumlah record per partition.
partition_profile = (
    df8.rdd
    .mapPartitionsWithIndex(lambda idx, it: [(idx, sum(1 for _ in it))])
    .collect()
)
partition_profile

[(0, 125000),
 (1, 125000),
 (2, 125000),
 (3, 125000),
 (4, 125000),
 (5, 125000),
 (6, 125000),
 (7, 125000)]

**Challenge:** Jika stage memiliki 8 task tetapi hanya 4 slot task tersedia, berapa gelombang minimal yang dibutuhkan bila durasi task seragam? Jelaskan hubungan partition, task, dan core.

## Lab 3.1 — Transformation, action (20 menit)
Jalankan cell pertama dan lihat Spark UI. Apakah job baru muncul? Baru setelah itu jalankan action.

In [64]:
base = spark.range(0, 2_000_000, 1, 4)
transformed = (
    base
    .filter((F.col("id") % 2) == 0)
    .withColumn("bucket", F.col("id") % 10)
    .select("id", "bucket")
)
print("DataFrame sudah didefinisikan. Belum ada action eksplisit.")

DataFrame sudah didefinisikan. Belum ada action eksplisit.


In [65]:
# ACTION: memicu komputasi
result_count = transformed.count()
print("Rows:", result_count)

Rows: 1000000


In [66]:
# ACTION: memicu komputasi
transformed.show()

+---+------+
| id|bucket|
+---+------+
|  0|     0|
|  2|     2|
|  4|     4|
|  6|     6|
|  8|     8|
| 10|     0|
| 12|     2|
| 14|     4|
| 16|     6|
| 18|     8|
| 20|     0|
| 22|     2|
| 24|     4|
| 26|     6|
| 28|     8|
| 30|     0|
| 32|     2|
| 34|     4|
| 36|     6|
| 38|     8|
+---+------+
only showing top 20 rows



**Pertanyaan:** `filter`, `withColumn`, `select`, `count` termasuk transformation atau action? Apa bukti yang terlihat pada UI?

## Lab 3.2 — Lazy evaluation

In [82]:
base_df = spark.range(0, 10_000_000, 1, 4)

filtered = base_df.filter(
    F.col("id") % 2 == 0
)

projected = filtered.select("id")

In [83]:
# Amati UI SEBELUM action ini:
answer = projected.count()
print("Hasil:", answer)

Hasil: 5000000


In [84]:
from pyspark.sql import functions as F

df = spark.range(0, 10_000_000, 1, 4)

result = (
    df
    .withColumn("group_id", F.col("id") % 10)
    .groupBy("group_id")
    .count()
)

result.show()


+--------+-------+
|group_id|  count|
+--------+-------+
|       0|1000000|
|       7|1000000|
|       6|1000000|
|       9|1000000|
|       5|1000000|
|       1|1000000|
|       3|1000000|
|       8|1000000|
|       2|1000000|
|       4|1000000|
+--------+-------+



## Lab 4 — Dari plan ke shuffle (25 menit)
`groupBy` adalah wide transformation. Cari `Exchange` pada physical plan dan cocokkan dengan stage boundary di UI.

In [85]:
agg = (
    transformed
    .groupBy("bucket")
    .agg(F.count("*").alias("n"), F.sum("id").alias("sum_id"))
    .orderBy("bucket")
)
agg.explain(True)

== Parsed Logical Plan ==
'Sort ['bucket ASC NULLS FIRST], true
+- Aggregate [bucket#162L], [bucket#162L, count(1) AS n#408L, sum(id#158L) AS sum_id#410L]
   +- Project [id#158L, bucket#162L]
      +- Project [id#158L, (id#158L % cast(10 as bigint)) AS bucket#162L]
         +- Filter ((id#158L % cast(2 as bigint)) = cast(0 as bigint))
            +- Range (0, 2000000, step=1, splits=Some(4))

== Analyzed Logical Plan ==
bucket: bigint, n: bigint, sum_id: bigint
Sort [bucket#162L ASC NULLS FIRST], true
+- Aggregate [bucket#162L], [bucket#162L, count(1) AS n#408L, sum(id#158L) AS sum_id#410L]
   +- Project [id#158L, bucket#162L]
      +- Project [id#158L, (id#158L % cast(10 as bigint)) AS bucket#162L]
         +- Filter ((id#158L % cast(2 as bigint)) = cast(0 as bigint))
            +- Range (0, 2000000, step=1, splits=Some(4))

== Optimized Logical Plan ==
Sort [bucket#162L ASC NULLS FIRST], true
+- Aggregate [bucket#162L], [bucket#162L, count(1) AS n#408L, sum(id#158L) AS sum_id#410L]


26/11/11 15:30:25 ERROR TaskSchedulerImpl: Lost executor 3 on 172.19.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/11/11 15:30:25 ERROR TaskSchedulerImpl: Lost executor 2 on 172.19.0.6: worker lost: Not receiving heartbeat for 60 seconds


In [75]:
agg.show(truncate=False)

+------+------+------------+
|bucket|n     |sum_id      |
+------+------+------------+
|0     |200000|199999000000|
|2     |200000|199999400000|
|4     |200000|199999800000|
|6     |200000|200000200000|
|8     |200000|200000600000|
+------+------+------------+



**Checklist Spark UI:**
- Job mana yang muncul karena `show()`?
- Berapa stage?
- Stage mana yang memiliki shuffle read/write?
- Berapa task pada masing-masing stage?

Jangan menganggap satu pemanggilan API = satu task.

## Lab 5 — Eksperimen jumlah partition (30 menit)
Ini **bukan benchmark ilmiah**. Tujuannya melihat trade-off parallelism vs scheduling overhead.

In [ ]:
rows = []
for n in [1, 2, 4, 8]:
    sample = spark.range(0, 10_000_000, 1, n)
    t0 = time.perf_counter()
    total = sample.agg(F.sum("id")).collect()[0][0]
    elapsed = time.perf_counter() - t0
    rows.append((n, round(elapsed, 3), total))

spark.createDataFrame(rows, ["partitions", "seconds", "sum_id"]).show(truncate=False)

**Expected checksum:** `49,999,995,000,000`.

Diskusikan:
- Mengapa 8 partition tidak selalu 8× lebih cepat daripada 1 partition?
- Apa yang terjadi jika jumlah task jauh lebih banyak daripada core?
- Mengapa pengukuran perlu diulang?

## Exit ticket
Tulis jawaban singkat:
1. Worker, Executor, Task, Partition: apa bedanya?
2. Apa yang memicu Job?
3. Mengapa `Exchange` penting saat membaca plan?
4. Mengapa `repartition()` bukan optimasi gratis?